In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/ML final project'

if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH)
    print(f"it created: {PROJECT_PATH}")

%cd {PROJECT_PATH}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/13YZMRWkS2eT6kmRi5e5vzlG7Yl3Vbt3z/ML final project


In [2]:
!pip install mlflow

In [3]:
!pip install dagshub


In [41]:
from google.colab import userdata
DAGSHUB_TOKEN = userdata.get('DAGSHUB_TOKEN').strip()

In [42]:
from transformers import WalmartDataTransformer, TimeSeriesSplitter

In [35]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/train.csv.zip')
features_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/features.csv.zip')
stores_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/stores.csv')

train_df['Date'] = pd.to_datetime(train_df['Date'])

train_raw_df = train_df[train_df['Date'] < '2012-01-01'].copy()
val_raw_df = train_df[train_df['Date'] >= '2012-01-01'].copy()

transformer = WalmartDataTransformer(features_df=features_df, stores_df=stores_df)
transformer.fit(train_raw_df, y=train_raw_df['Weekly_Sales'])

train_processed = transformer.transform(train_raw_df)
val_processed = transformer.transform(val_raw_df)

train_processed['Is_Pre_Holiday'] = train_processed.groupby(['Store', 'Dept'])['IsHoliday'].shift(-1).fillna(0).astype(int)
train_processed['Is_Post_Holiday'] = train_processed.groupby(['Store', 'Dept'])['IsHoliday'].shift(1).fillna(0).astype(int)

val_processed['Is_Pre_Holiday'] = val_processed.groupby(['Store', 'Dept'])['IsHoliday'].shift(-1).fillna(0).astype(int)
val_processed['Is_Post_Holiday'] = val_processed.groupby(['Store', 'Dept'])['IsHoliday'].shift(1).fillna(0).astype(int)

In [49]:
import importlib
import transformers
importlib.reload(transformers)
from transformers import WalmartDataTransformer

In [50]:
import pickle
import pandas as pd
import numpy as np

with open("walmart_lgbm_best_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

print("🚀 Model loaded:", loaded_model)

train_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/train.csv.zip')
test_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/test.csv.zip')
features_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/features.csv.zip')
stores_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/stores.csv')

train_df['Date'] = pd.to_datetime(train_df['Date'])
test_df['Date'] = pd.to_datetime(test_df['Date'])

full_df = pd.concat([train_df, test_df], ignore_index=True)

from transformers import WalmartDataTransformer

transformer = WalmartDataTransformer(features_df=features_df, stores_df=stores_df)
transformer.fit(train_df, y=train_df['Weekly_Sales'])
full_processed = transformer.transform(full_df)

test_processed = full_processed[full_processed['Date'] >= '2012-11-02'].copy()

features_to_use = [col for col in test_processed.columns if col not in ['Date', 'Weekly_Sales', 'Year']]
X_test_final = test_processed[features_to_use].copy()

for col in X_test_final.columns:
    if X_test_final[col].dtype == 'object':
        X_test_final[col] = X_test_final[col].astype('category')

model_features = list(loaded_model.feature_name_)
missing_in_test = set(model_features) - set(X_test_final.columns)
extra_in_test = set(X_test_final.columns) - set(model_features)

if missing_in_test:
    print(f"⚠️ Columns model expects but test is missing: {missing_in_test}")
if extra_in_test:
    print(f"⚠️ Extra columns in test not seen during training: {extra_in_test}")
if not missing_in_test and not extra_in_test:
    print("✅ Feature columns match training exactly.")

X_test_final = X_test_final[model_features]

nan_columns = X_test_final.isna().sum()
nan_columns = nan_columns[nan_columns > 0]
if len(nan_columns) > 0:
    print("⚠️ NaNs found, filling with 0:")
    print(nan_columns)
    X_test_final = X_test_final.fillna(0)
else:
    print("✅ No NaNs in feature matrix.")

y_test_pred = loaded_model.predict(X_test_final)

submission_df = pd.DataFrame()
submission_df['Id'] = (
    test_processed['Store'].astype(str) + '_' +
    test_processed['Dept'].astype(str) + '_' +
    test_processed['Date'].astype(str)
)
submission_df['Weekly_Sales'] = y_test_pred

output_filename = 'lgbm_final_champion_submission.csv'
submission_df.to_csv(output_filename, index=False)

print(f"\n🚀 Submission saved: {output_filename}")
print(f"📊 Rows: {len(submission_df)}")
print(submission_df.head())

🚀 Model loaded: LGBMRegressor(bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.75,
              learning_rate=0.03, min_data_in_leaf=10, n_estimators=2000,
              n_jobs=-1, num_leaves=255, objective='mae', random_state=42,
              reg_alpha=10.0, reg_lambda=10.0, verbose=-1)
✅ Feature columns match training exactly.
✅ No NaNs in feature matrix.

🚀 Submission saved: lgbm_final_champion_submission.csv
📊 Rows: 115064
                 Id  Weekly_Sales
143  1_1_2012-11-02  38181.059830
144  1_1_2012-11-09  17879.083358
145  1_1_2012-11-16  17574.984153
146  1_1_2012-11-23  19385.538956
147  1_1_2012-11-30  23198.288073
